In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

In [ ]:
DATA_PATH = "Projet_Data/processed/iot_preprocessed.csv"
LOOKBACK = 24          
BATCH_SIZE = 256
EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 4           
HIDDEN = 128
DROPOUT = 0.2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration prête.")

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values("time").reset_index(drop=True)

targets = ['target_temp', 'target_humidity', 'target_co2']
excluded_columns = ["time", "deveui"] + targets
features = [col for col in df.columns if col not in excluded_columns]

print(f"Nombre de features : {len(features)}")
print(f"Nombre de cibles : {len(targets)} (Multi-sorties)")

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

x_scaler = StandardScaler()
train_X = x_scaler.fit_transform(train_df[features].values.astype(np.float32))
test_X = x_scaler.transform(test_df[features].values.astype(np.float32))

y_scaler = StandardScaler()
train_y_s = y_scaler.fit_transform(train_df[targets].values.astype(np.float32))
test_y_s = y_scaler.transform(test_df[targets].values.astype(np.float32))

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, lookback):
        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32))
        self.lookback = lookback

    def __len__(self):
        return len(self.X) - self.lookback + 1

    def __getitem__(self, idx):
        x = self.X[idx : idx + self.lookback]
        y = self.y[idx + self.lookback - 1]
        return x, y

train_ds = SequenceDataset(train_X, train_y_s, LOOKBACK)
test_ds = SequenceDataset(test_X, test_y_s, LOOKBACK)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

n_features = train_X.shape[1]
n_outputs = len(targets)

In [ ]:

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.LayerNorm(dim)
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class ResidualMLP(nn.Module):
    def __init__(self, n_features, lookback, hidden=128, dropout=0.2):
        super().__init__()
        in_dim = n_features * lookback
        self.input = nn.Sequential(
            nn.Linear(in_dim, hidden * 2), nn.LayerNorm(hidden * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        self.blocks = nn.Sequential(ResidualBlock(hidden, dropout), ResidualBlock(hidden, dropout))
        self.head = nn.Linear(hidden, n_outputs) # 3 Sorties
    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        x = self.input(x)
        return self.head(self.blocks(x))

class TemporalCNN(nn.Module):
    def __init__(self, n_features, hidden=128, dropout=0.2):
        super().__init__()
        channels = [n_features, hidden, hidden, hidden]
        dilations = [1, 2, 4]
        layers = []
        for i, d in enumerate(dilations):
            layers += [
                nn.Conv1d(channels[i], channels[i + 1], kernel_size=3, padding=d, dilation=d),
                nn.BatchNorm1d(channels[i + 1]), nn.GELU(), nn.Dropout(dropout)
            ]
        self.conv = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, n_outputs)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        return self.head(self.conv(x))

class StackedLSTM(nn.Module):
    def __init__(self, n_features, hidden=128, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=2, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden // 2, n_outputs)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

class BiGRU(nn.Module):
    def __init__(self, n_features, hidden=128, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, num_layers=2, batch_first=True, dropout=dropout, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_outputs)
        )
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])


class TimeSeriesTransformer(nn.Module):
    def __init__(self, n_features, hidden=128, n_heads=4, dropout=0.2):
        super().__init__()
        self.embedding = nn.Linear(n_features, hidden)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_outputs) # 3 sorties (Temp, Hum, CO2)
        )

    def forward(self, x):
        x = self.embedding(x)
        out = self.transformer(x)
        return self.head(out[:, -1, :])


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model_builders = {
    "Residual MLP": lambda: ResidualMLP(n_features, LOOKBACK, HIDDEN, DROPOUT),
    "Temporal CNN": lambda: TemporalCNN(n_features, HIDDEN, DROPOUT),
    "Stacked LSTM": lambda: StackedLSTM(n_features, HIDDEN, DROPOUT),
    "Bidirectional GRU": lambda: BiGRU(n_features, HIDDEN, DROPOUT),
    "Transformer": lambda: TimeSeriesTransformer(n_features, HIDDEN, 4, DROPOUT)
}

for name, builder in model_builders.items():
    print(f"{name:22s}  params={count_params(builder()):>10,}")

In [ ]:

def train_one_model(model, train_loader, test_loader, epochs=EPOCHS, patience=PATIENCE):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        model.train()
        tr_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                va_losses.append(loss_fn(model(xb), yb).item())

        tr, va = float(np.mean(tr_losses)), float(np.mean(va_losses))
        sched.step(va)
        print(f"  epoch {epoch:02d}/{epochs}  train_mse={tr:.5f}  val_mse={va:.5f}")

        if va < best_val - 1e-6:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

def get_average_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred, multioutput='uniform_average')
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average')))
    r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
    return {"MAE Moyenne": mae, "RMSE Moyenne": rmse, "R2 Moyen": r2}

In [ ]:
results = {}
os.makedirs("Projet_Data/models_dl", exist_ok=True)

for name, builder in model_builders.items():
    print(f"\nEntraînement: {name}")
    print("=" * 40)
    model = builder()
    model = train_one_model(model, train_loader, test_loader)
    
    y_pred_s, y_true_s = predict(model, test_loader)
    
    y_pred = y_scaler.inverse_transform(y_pred_s)
    y_true = y_scaler.inverse_transform(y_true_s)

    metrics = get_average_metrics(y_true, y_pred)
    results[name] = metrics
    
    torch.save(model.state_dict(), f"Projet_Data/models_dl/{name.replace(' ', '_').lower()}.pt")

print("\n--- Bilan des modèles Deep Learning ---")
display(pd.DataFrame(results).T.sort_values(by="R2 Moyen", ascending=False))

In [ ]:
import pandas as pd

print("--- Bilan final des modèles Deep Learning ---")
df_results = pd.DataFrame(results).T.sort_values(by="R2 Moyen", ascending=False)
display(df_results)

In [ ]:
def afficher_prediction_future(nom_meilleur_modele, df, features, targets, x_scaler, y_scaler, lookback):
    best_model = model_builders[nom_meilleur_modele]()
    best_model.load_state_dict(torch.load(f"Projet_Data/models_dl/{nom_meilleur_modele.replace(' ', '_').lower()}.pt"))
    best_model = best_model.to(DEVICE)
    best_model.eval()

    last_data = df[features].tail(lookback).values
    
    colonnes_originales = ['temperature', 'humidity', 'co2']
    etat_actuel = df[colonnes_originales].tail(1).values[0]

    last_data_scaled = x_scaler.transform(last_data).astype(np.float32)
    x_tensor = torch.tensor(last_data_scaled).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_scaled = best_model(x_tensor).cpu().numpy()
        
    pred_future = y_scaler.inverse_transform(pred_scaled)[0]

    print("-" * 50)
    print("SYSTÈME DE PRÉDICTION ENVIRONNEMENTALE")
    print("-" * 50)
    print(f"État actuel : Température = {etat_actuel[0]:.1f} °C | Humidité = {etat_actuel[1]:.1f} % | CO2 = {etat_actuel[2]:.0f} ppm")
    print("\nLe modèle prédit :")
    print(f"Dans 30 minutes : Température = {pred_future[0]:.1f} °C, Humidité = {pred_future[1]:.1f} %, CO2 = {pred_future[2]:.0f} ppm.")
    print("-" * 50)

afficher_prediction_future("Temporal CNN", df, features, targets, x_scaler, y_scaler, LOOKBACK)

In [ ]:

import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np

print("Génération de la Figure 4.2 (Loss)...")
model_cnn = TemporalCNN(n_features, HIDDEN, DROPOUT).to(DEVICE)
opt = optim.AdamW(model_cnn.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

train_losses, val_losses = [], []

# Entraînement rapide sur 10 époques pour capturer la Loss
for epoch in range(10): 
    model_cnn.train()
    batch_train_loss = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model_cnn(xb), yb)
        loss.backward()
        opt.step()
        batch_train_loss.append(loss.item())
    train_losses.append(np.mean(batch_train_loss))
    
    model_cnn.eval()
    batch_val_loss = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            batch_val_loss.append(loss_fn(model_cnn(xb), yb).item())
    val_losses.append(np.mean(batch_val_loss))

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='o')
plt.title("Figure 4.2 : Convergence du Temporal CNN (Loss)")
plt.xlabel("Époques")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
import torch
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

print("\nGénération de la Figure 4.3 (Prédictions vs Réalité)...")
best_model = model_builders["Temporal CNN"]()
best_model.load_state_dict(torch.load("Projet_Data/models_dl/temporal_cnn.pt"))
best_model = best_model.to(DEVICE)

y_pred_s, y_true_s = predict(best_model, test_loader)
y_pred = y_scaler.inverse_transform(y_pred_s)
y_true = y_scaler.inverse_transform(y_true_s)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))
variables = ['Température', 'Humidité', 'CO2']
for i in range(3):
    # On affiche les 200 premières prédictions de test
    axes[i].plot(y_true[:200, i], label=f'{variables[i]} Réelle', alpha=0.7)
    axes[i].plot(y_pred[:200, i], label=f'Prédiction CNN (t+30min)', linestyle='--')
    axes[i].set_title(f"Figure 4.3 : Prédiction à t+30min - {variables[i]}")
    axes[i].legend()
plt.tight_layout()
import os
os.makedirs("Projet_Data/images", exist_ok=True)
plt.savefig("Projet_Data/images/figure_9.png", bbox_inches='tight', dpi=300)
print("Graphique sauvegardé dans Projet_Data/images/figure_9.png")

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

variables = ['Température', 'Humidité', 'CO2']
metrics_per_var = {var: {"MAE": {}, "MSE": {}, "RMSE": {}} for var in variables}

print("Calcul des métriques individuelles en cours...")
for name, builder in model_builders.items():
    model = builder()
    model.load_state_dict(torch.load(f"Projet_Data/models_dl/{name.replace(' ', '_').lower()}.pt"))
    model = model.to(DEVICE)
    
    y_pred_s, y_true_s = predict(model, test_loader)
    y_pred = y_scaler.inverse_transform(y_pred_s)
    y_true = y_scaler.inverse_transform(y_true_s)
    
    for i, var in enumerate(variables):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        mse = mean_squared_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        
        metrics_per_var[var]["MAE"][name] = mae
        metrics_per_var[var]["MSE"][name] = mse
        metrics_per_var[var]["RMSE"][name] = rmse

for var in variables:
    print(f"\n--- Résultats pour : {var} ---")
    df_var = pd.DataFrame(metrics_per_var[var]).sort_values(by="RMSE")
    print(df_var.to_string()) # ou print(df_var.to_string())
    df_co2 = pd.DataFrame(metrics_per_var['CO2']).sort_values(by="RMSE")
    print("--- Résultats pour : CO2 ---")
    print(df_co2.to_string())